# Laplace and CVI: A Poisson Count Series

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. *Laplace Approximation: Stochastic Volatility* smoothed a latent autoregressive process observed through a nonlinear node. Here the observations are counts, and we compare the two Gaussian projections the fallback offers, `LaplaceApproximation()` and `CVI()`, with the exact posterior.

A latent log-rate follows an autoregressive process, and each count is Poisson with rate $e^{x_t}$:

$$\begin{aligned}
x_1 &\sim \mathcal{N}(0.5, 1),\\
x_t &\sim \mathcal{N}(\varphi x_{t-1}, \sigma^2),\\
y_t &\sim \mathrm{Poisson}(e^{x_t}).
\end{aligned}$$

Because the state is one-dimensional, the exact posterior is within reach: on a fine grid of $x$, the model is a hidden Markov model, and the forward-backward algorithm gives the exact smoothing marginals and the exact evidence $p(y)$.

In [ ]:
using RxInfer, BayesBase, Distributions, SpecialFunctions, StableRNGs
using Plots, PrettyTables, Printf

## The observation node

The count is a node of its own, defined by its log-density alone. Writing it as `y[t] ~ Poisson(exp(x[t]))` would put a deterministic node `exp` before RxInfer's `Poisson` node; the fallback's message from such a node is a Gaussian on the whole real line, which knows nothing of the rate being positive, and Poisson's rules need a positive rate. A node that takes the log-rate directly has no such problem.

In [ ]:
struct LogPoisson{X} <: DiscreteUnivariateDistribution
    x::X
end

BayesBase.logpdf(d::LogPoisson, k) = k * d.x - exp(d.x) - loggamma(k + 1)
BayesBase.insupport(::LogPoisson, k) = true

@define_factor_node(node = LogPoisson, type = Stochastic, interfaces = [:out, :x])

## Data

In [ ]:
φ, σ², T = 0.9, 0.1, 150
rng = StableRNG(42)

x_true = zeros(T)
x_true[1] = rand(rng, Normal(0.5, 1.0))
for t in 2:T
    x_true[t] = φ * x_true[t - 1] + rand(rng, Normal(0.0, sqrt(σ²)))
end
y = [rand(rng, Poisson(exp(x))) for x in x_true]

bar(y; color = :gray, label = "counts yₜ", xlabel = "t", size = (800, 300))

## Inference

In [ ]:
@model function count_series(y, φ, σ²)
    x[1] ~ NormalMeanVariance(0.5, 1.0)
    y[1] ~ LogPoisson(x[1])
    for t in 2:length(y)
        x[t] ~ NormalMeanVariance(φ * x[t - 1], σ²)
        y[t] ~ LogPoisson(x[t])
    end
end

fit(fallback) = infer(
    model = count_series(φ = φ, σ² = σ²), data = (y = y,),
    fallback = fallback, free_energy = true, report = true,
)

laplace = fit(LaplaceApproximation())
cvi = fit(CVI())

Under belief propagation the transitions keep their exact rules. Every product of messages on the chain, a marginal or a message leaving $x_t$, is projected onto a Gaussian: by the Laplace method at its mode, or by `CVI()` onto the Gaussian closest to it in the sense of $\mathrm{KL}(q \,\|\, p)$. The report of the CVI run shows it:

In [ ]:
@assert isempty(RxInfer.diagnostics(laplace.report)) && isempty(RxInfer.diagnostics(cvi.report)) # hide
@assert RxInfer.node_types(cvi.report.approximations)[(LogPoisson, :message, :x)].fallbacks == T # hide
cvi.report.approximations

## The exact smoother

On a grid of 801 points for $x$, the transition is a matrix and each count a vector of likelihoods. A forward pass gives the filtering distributions and the evidence, and a backward pass the smoothing ones.

In [ ]:
function exact_smoother(y; grid = range(-5, 5, length = 801))
    h = step(grid)
    K = [pdf(Normal(φ * u, sqrt(σ²)), v) * h for u in grid, v in grid]
    likelihood(t) = [exp(logpdf(LogPoisson(u), y[t])) for u in grid]
    α, c = zeros(length(grid), T), zeros(T)
    a = [pdf(Normal(0.5, 1.0), u) * h for u in grid] .* likelihood(1)
    c[1] = sum(a); α[:, 1] = a ./ c[1]
    for t in 2:T
        a = (K' * α[:, t - 1]) .* likelihood(t)
        c[t] = sum(a); α[:, t] = a ./ c[t]
    end
    β = ones(length(grid), T)
    for t in (T - 1):-1:1
        β[:, t] = K * (likelihood(t + 1) .* β[:, t + 1]) ./ c[t + 1]
    end
    P = α .* β
    P ./= sum(P; dims = 1)
    m = [sum(P[:, t] .* grid) for t in 1:T]
    s = [sqrt(sum(P[:, t] .* (grid .- m[t]) .^ 2)) for t in 1:T]
    return (; mean = m, sd = s, logevidence = sum(log, c))
end

exact = exact_smoother(y)

## Results

In [ ]:
m_laplace, s_laplace = mean.(laplace.posteriors[:x]), std.(laplace.posteriors[:x])
m_cvi, s_cvi = mean.(cvi.posteriors[:x]), std.(cvi.posteriors[:x])

plt = plot(exact.mean; ribbon = 2exact.sd, fillalpha = 0.2, color = :gray, label = "exact (±2 sd)", xlabel = "t", ylabel = "log-rate xₜ", size = (900, 450), legend = :outerbottom, legendcolumns = 4)
plot!(plt, m_laplace; color = :blue, label = "LaplaceApproximation()")
plot!(plt, m_cvi; color = :red, linestyle = :dash, label = "CVI()")
plot!(plt, x_true; color = :black, linewidth = 1, label = "true xₜ")

In [ ]:
rms(a) = sqrt(sum(abs2, a) / length(a))
F_laplace, F_cvi = only(laplace.free_energy), only(cvi.free_energy)
rows = [
    "LaplaceApproximation()" rms(m_laplace - exact.mean) maximum(abs, m_laplace - exact.mean) maximum(abs, s_laplace - exact.sd) F_laplace;
    "CVI()" rms(m_cvi - exact.mean) maximum(abs, m_cvi - exact.mean) maximum(abs, s_cvi - exact.sd) F_cvi;
    "exact" 0.0 0.0 0.0 -exact.logevidence
]
@assert rms(m_cvi - exact.mean) < 0.5 * rms(m_laplace - exact.mean) # hide
@assert maximum(abs, s_cvi - exact.sd) < 0.05 && maximum(abs, s_laplace - exact.sd) < 0.05 # hide
@assert -exact.logevidence < F_laplace < F_cvi # hide
pretty_table(rows; column_labels = ["", "RMS of mean - exact", "max |mean - exact|", "max |sd - exact|", "free energy"], formatters = [fmt__printf("%.4f", 2:5)], fit_table_in_display_horizontally = false)

`CVI()`'s means are several times closer to the exact smoother than Laplace's. A count of zero, or a count far from the rate, makes the posterior of $x_t$ skewed; Laplace places its Gaussian at the mode, `CVI()` near the mean. The spreads of both are close to the exact ones.

The free energy ranks them the other way, and both exceed $-\log p(y)$. Under belief propagation, a projected marginal and the joints of the transition nodes around it are computed from different messages, so they do not agree exactly. The free energy of such beliefs is then an estimate of $-\log p(y)$, not a bound, and a lower one is not a better fit. Making them agree is what expectation propagation does, and it is not part of universal inference yet. Under mean field, where every belief comes from the same marginals, the free energy is a bound again, as in *Laplace and CVI: A Hierarchical Model*.

## Learning the transition precision

Now the precision of the transition, $\tau = 1/\sigma^2$, is unknown, with a $\mathrm{Gamma}(2, 0.2)$ prior, and we factorise $q(x, \tau) = q(x)\,q(\tau)$. The chain keeps its belief propagation, and $\tau$'s update is conjugate: its posterior is a Gamma distribution, which passes the fallback's projection unchanged.

In [ ]:
@model function count_series_τ(y, φ)
    τ ~ Gamma(shape = 2.0, rate = 0.2)
    x[1] ~ NormalMeanVariance(0.5, 1.0)
    y[1] ~ LogPoisson(x[1])
    for t in 2:length(y)
        x[t] ~ NormalMeanPrecision(φ * x[t - 1], τ)
        y[t] ~ LogPoisson(x[t])
    end
end

separate_τ = @constraints begin
    q(x, τ) = q(x)q(τ)
end

initial_τ = @initialization begin
    q(τ) = GammaShapeRate(2.0, 0.2)
end

fit_τ(fallback) = infer(
    model = count_series_τ(φ = φ), data = (y = y,),
    constraints = separate_τ, initialization = initial_τ,
    iterations = 30, fallback = fallback, free_energy = true, report = true,
)

laplace_τ, cvi_τ = fit_τ(LaplaceApproximation()), fit_τ(CVI())

The report confirms that $\tau$'s posterior is exact and every $x_t$'s is projected:

In [ ]:
records_τ = cvi_τ.report.approximations
@assert !RxInfer.is_approximated(records_τ.variables[(:τ, nothing)]) # hide
@assert count(RxInfer.is_approximated, values(records_τ.variables)) == T # hide
@assert isempty(RxInfer.diagnostics(cvi_τ.report)) && isempty(RxInfer.diagnostics(laplace_τ.report)) # hide
records_τ

For the exact posterior of $\tau$, the forward pass gives the evidence $p(y \mid \tau)$ on a grid of $\tau$ values, a coarser grid of $x$ being enough for it.

In [ ]:
function logevidence_given(τ; grid = range(-4, 4, length = 321))
    h = step(grid)
    K = [pdf(Normal(φ * u, 1 / sqrt(τ)), v) * h for u in grid, v in grid]
    a = [pdf(Normal(0.5, 1.0), u) * h for u in grid] .* [exp(logpdf(LogPoisson(u), y[1])) for u in grid]
    s = log(sum(a)); a ./= sum(a)
    for t in 2:T
        a = (K' * a) .* [exp(logpdf(LogPoisson(u), y[t])) for u in grid]
        z = sum(a); s += log(z); a ./= z
    end
    return s
end

τs = range(0.5, 60, length = 240)
logposterior_τ = [logpdf(Gamma(2.0, 1 / 0.2), τ) + logevidence_given(τ) for τ in τs]
w = exp.(logposterior_τ .- maximum(logposterior_τ)); w ./= sum(w)
Eτ = sum(w .* τs); sdτ = sqrt(sum(w .* (τs .- Eτ) .^ 2))

qτ_laplace, qτ_cvi = laplace_τ.posteriors[:τ][end], cvi_τ.posteriors[:τ][end]
@assert Eτ > 1 / σ² && max(std(qτ_laplace), std(qτ_cvi)) < 0.4 * sdτ # hide
@assert abs(mean(qτ_laplace) - Eτ) < 0.5 * sdτ && abs(mean(qτ_cvi) - Eτ) < 0.5 * sdτ # hide
pretty_table(
    ["exact" Eτ sdτ; "LaplaceApproximation()" mean(qτ_laplace) std(qτ_laplace); "CVI()" mean(qτ_cvi) std(qτ_cvi)];
    column_labels = ["", "E[τ]", "sd(τ)"], formatters = [fmt__printf("%.2f", 2:3)], fit_table_in_display_horizontally = false,
)

The data, 150 counts of about one or two, say little about $\tau$: its exact posterior is wide, and its mean is above the true value 10. Both schemes find a mean within half a standard deviation of the exact one, and both are overconfident by a factor of three or more. That is the price of the factorisation $q(x)\,q(\tau)$: the path and its precision are learned as if independent, while a rougher path and a lower precision explain the same counts. The plot shows the exact posterior and the two Gamma approximations.

In [ ]:
τplot = range(0.5, 45, length = 400)
plot(τs, w ./ step(τs); color = :black, linewidth = 2, label = "exact", xlabel = "τ", ylabel = "density", xlims = (0, 45))
plot!(τplot, pdf.(qτ_laplace, τplot); color = :blue, label = "LaplaceApproximation()")
plot!(τplot, pdf.(qτ_cvi, τplot); color = :red, linestyle = :dash, label = "CVI()")
vline!([1 / σ²]; color = :gray, linestyle = :dot, label = "true τ")